# Feature selection: recorded death

Compare logistic regression, decision-tree and Boruta selections of **20 original attributes each**.
Run `python src/select_features.py` from the project folder first. All methods use the
same training/test split and validation folds. This is an exploratory association
analysis; importance does not establish a cause of death. Target coding 1=death
must be checked against the data dictionary.


In [6]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

project = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
runs = sorted((project / "results" / "feature_selection").glob("*/config.json"))
if not runs:
    raise FileNotFoundError("Run python src/select_features.py first.")
run = runs[-1].parent  # Change this path to compare a particular saved run.
config = json.loads((run / "config.json").read_text())
selected = json.loads((run / "selected_features.json").read_text())
print("Run:", run.name)
print(config["scope"])
print("Training patients:", config["train_rows"], "Test patients:", config["test_rows"])


Run: 20261101T122557_528697Z
Exploratory association with recorded death, not causal attribution or fixed-horizon risk.
Training patients: 314 Test patients: 79


## Compare model performance

**ROC AUC is the primary metric.** Compare methods by mean cross-validation ROC AUC
and show the standard deviation across folds. Each fold repeats feature selection
on its training patients. Final test ROC AUC is reported separately and is not
used to choose the method. The remaining metrics provide supporting information.


In [7]:
cv_summary = pd.read_csv(run / "cv_summary.csv").sort_values(
    "roc_auc_mean", ascending=False
)
test_results = (
    pd.read_csv(run / "test_results.csv")
    .set_index("method")
    .loc[cv_summary["method"]]
    .reset_index()
)
print("Primary comparison: cross-validation ROC AUC")
display(cv_summary[["method", "roc_auc_mean", "roc_auc_std"]])
print("Held-out test ROC AUC (reporting only)")
display(test_results[["method", "roc_auc"]])
print("Supporting metrics")
display(cv_summary)
display(test_results)


Primary comparison: cross-validation ROC AUC


,method,roc_auc_mean,roc_auc_std
0,boruta,0.768066,0.027436
1,decision_tree,0.656668,0.081159
2,logistic_regression,0.589012,0.086578


Held-out test ROC AUC (reporting only)


,method,roc_auc
0,boruta,0.781699
1,decision_tree,0.630719
2,logistic_regression,0.737255


Supporting metrics


,method,roc_auc_mean,roc_auc_std,average_precision_mean,average_precision_std,balanced_accuracy_mean,balanced_accuracy_std,accuracy_mean,accuracy_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std
0,boruta,0.768066,0.027436,0.841308,0.020922,0.731947,0.031019,0.729186,0.026680,0.800011,0.050583,0.712613,0.042751,0.752063,0.021325
1,decision_tree,0.656668,0.081159,0.700288,0.054867,0.640870,0.053201,0.643164,0.055797,0.705944,0.045933,0.657808,0.105064,0.677425,0.062168
2,logistic_regression,0.589012,0.086578,0.686195,0.067408,0.557409,0.068517,0.554173,0.073292,0.631479,0.064865,0.536186,0.115162,0.577285,0.085889


,method,n_features,roc_auc,average_precision,balanced_accuracy,accuracy,precision,recall,f1,true_negative,false_positive,false_negative,true_positive
0,boruta,20,0.781699,0.773702,0.693791,0.696203,0.744186,0.711111,0.727273,23,11,13,32
1,decision_tree,20,0.630719,0.659389,0.623529,0.620253,0.692308,0.600000,0.642857,22,12,18,27
2,logistic_regression,20,0.737255,0.765701,0.682680,0.683544,0.738095,0.688889,0.712644,23,11,14,31


## Compare the selected attributes

Each name represents an original column, even when preprocessing creates several
one-hot columns. Logistic, tree and Boruta scores have different scales and should not
be compared directly. Tree impurity importance can favor variables with many
possible splits. Zero-score ties are filled alphabetically to keep the requested
20 attributes; those names do not have evidence of importance from that fit.


In [ ]:
display(pd.DataFrame(selected))
from itertools import combinations

for left, right in combinations(selected, 2):
    shared = sorted(set(selected[left]) & set(selected[right]))
    print(f"{left} / {right}: {len(shared)} shared attributes:", shared)
shared_all = sorted(set.intersection(*(set(names) for names in selected.values())))
print("Shared by every method:", shared_all)
rankings = pd.read_csv(run / "feature_rankings.csv")
for method in selected:
    print(method)
    display(rankings.loc[(rankings["method"] == method) & rankings["selected"]])


,logistic_regression,decision_tree,boruta
0,Wskazania,B_NT - proBNP,B_NT - proBNP
1,typo urządzenia,MDRDcorr,LVEF
2,Es'_mean,"TR_ica,ICA,PSV-ICA cm/s",Ea'_mean
3,producent,"TEB,0,PEP mean",B_Kreatynina
4,"TL_cca,CCA,PI-CCA",Ea'_mean,E_diastarea
5,"E_Tylna ściana lk,",LVEF,egfr_CG
6,Ant_MHG_R,"TL_cca,CCA,EDV-CCA cm/s",MDRDcorr
7,"TL_ica,ICA,PSV-ICA cm/s","6mwt_,rr_,RR1 minuta DBP",E_LV2j rozk cm2
8,B_Potas,"TEB,0,LCW mean",Edoeprim
9,Edoeprim,Ant_Rytm serca,"E_rozk,LAA cm2.1"


logistic_regression / decision_tree: 3 shared attributes: ['Ant_DBP', 'B_NT - proBNP', 'B_PLT']
logistic_regression / boruta: 3 shared attributes: ['B_NT - proBNP', 'Edoeprim', 'TL_ica,ICA,PSV-ICA cm/s']
decision_tree / boruta: 5 shared attributes: ['B_NT - proBNP', 'E_rozk,RAA cm2', "Ea'_mean", 'LVEF', 'MDRDcorr']
Shared by every method: ['B_NT - proBNP']
logistic_regression


,method,rank,feature,importance,selected,boruta_rank,boruta_status
0,logistic_regression,1,Wskazania,1.113817,True,NaN,NaN
1,logistic_regression,2,typo urządzenia,0.981071,True,NaN,NaN
2,logistic_regression,3,Es'_mean,0.967733,True,NaN,NaN
3,logistic_regression,4,producent,0.956587,True,NaN,NaN
4,logistic_regression,5,"TL_cca,CCA,PI-CCA",0.906851,True,NaN,NaN
5,logistic_regression,6,"E_Tylna ściana lk,",0.879365,True,NaN,NaN
6,logistic_regression,7,Ant_MHG_R,0.848063,True,NaN,NaN
7,logistic_regression,8,"TL_ica,ICA,PSV-ICA cm/s",0.800178,True,NaN,NaN
8,logistic_regression,9,B_Potas,0.781667,True,NaN,NaN
9,logistic_regression,10,Edoeprim,0.753331,True,NaN,NaN


decision_tree


,method,rank,feature,importance,selected,boruta_rank,boruta_status
287,decision_tree,1,B_NT - proBNP,0.303318,True,NaN,NaN
288,decision_tree,2,MDRDcorr,0.081884,True,NaN,NaN
289,decision_tree,3,"TR_ica,ICA,PSV-ICA cm/s",0.070881,True,NaN,NaN
290,decision_tree,4,"TEB,0,PEP mean",0.064213,True,NaN,NaN
291,decision_tree,5,Ea'_mean,0.052663,True,NaN,NaN
292,decision_tree,6,LVEF,0.051981,True,NaN,NaN
293,decision_tree,7,"TL_cca,CCA,EDV-CCA cm/s",0.051446,True,NaN,NaN
294,decision_tree,8,"6mwt_,rr_,RR1 minuta DBP",0.048457,True,NaN,NaN
295,decision_tree,9,"TEB,0,LCW mean",0.046767,True,NaN,NaN
296,decision_tree,10,Ant_Rytm serca,0.044142,True,NaN,NaN


boruta


,method,rank,feature,importance,selected,boruta_rank,boruta_status
574,boruta,1,B_NT - proBNP,0.085473,True,1.0,confirmed
575,boruta,2,LVEF,0.030783,True,1.0,confirmed
576,boruta,3,Ea'_mean,0.030109,True,1.0,confirmed
577,boruta,4,B_Kreatynina,0.028327,True,1.0,confirmed
578,boruta,5,E_diastarea,0.028255,True,1.0,confirmed
579,boruta,6,egfr_CG,0.027774,True,1.0,confirmed
580,boruta,7,MDRDcorr,0.026274,True,1.0,confirmed
581,boruta,8,E_LV2j rozk cm2,0.025355,True,1.0,confirmed
582,boruta,9,Edoeprim,0.022064,True,1.0,confirmed
583,boruta,10,"E_rozk,LAA cm2.1",0.021620,True,1.0,confirmed


## Interpreting Boruta

Boruta compares real attributes against shuffled shadow features using random
forests. It reports confirmed, tentative and rejected encoded features. We map
these to original attributes: any confirmed category makes its original
attribute confirmed; otherwise any tentative category makes it tentative.
This mapping is a reporting convention, not a joint statistical test.

For this comparison, we still take exactly 20 original attributes: best Boruta
rank first, then summed median importance, then name. Read `boruta_status` in
the ranking table: the 20 selected names are not necessarily 20 confirmed names.
Boruta's selection is evaluated with a random-forest classifier. ROC AUC scores
therefore compare selector/classifier combinations, not selectors alone.


In [4]:
boruta_rankings = pd.read_csv(run / "feature_rankings.csv")
boruta_rankings = boruta_rankings.loc[boruta_rankings["method"] == "boruta"].copy()

if boruta_rankings.empty or "boruta_status" not in boruta_rankings:
    print("This saved run has no Boruta results. Run src/select_features.py again.")
else:
    confirmed = boruta_rankings.loc[boruta_rankings["boruta_status"] == "confirmed"]
    print(f"BORUTA CONFIRMED {len(confirmed)} RELEVANT ORIGINAL ATTRIBUTES")
    print("Counts below refer to the final fit on training patients, not the validation folds.")
    status_counts = (
        boruta_rankings["boruta_status"]
        .value_counts()
        .reindex(["confirmed", "tentative", "rejected"], fill_value=0)
        .rename_axis("Boruta status")
        .to_frame("Original attributes")
    )
    display(status_counts)
    comparison = boruta_rankings.loc[boruta_rankings["selected"]]
    confirmed_in_comparison = int((comparison["boruta_status"] == "confirmed").sum())
    tentative_in_comparison = int((comparison["boruta_status"] == "tentative").sum())
    rejected_in_comparison = int((comparison["boruta_status"] == "rejected").sum())
    print(
        f"The {len(comparison)}-attribute comparison uses {confirmed_in_comparison} confirmed, "
        f"{tentative_in_comparison} tentative and {rejected_in_comparison} rejected attributes."
    )
    print("Tentative and rejected attributes are not confirmed relevant.")
    print("Confirmed relevant attributes:")
    display(confirmed[["feature", "boruta_rank", "importance"]].reset_index(drop=True))


BORUTA CONFIRMED 17 RELEVANT ORIGINAL ATTRIBUTES
Counts below refer to the final fit on training patients, not the validation folds.
               Original attributes
Boruta status                     
confirmed                       17
tentative                        8
rejected                       262
The 20-attribute comparison uses 17 confirmed, 3 tentative and 0 rejected attributes.
Tentative and rejected attributes are not confirmed relevant.
Confirmed relevant attributes:
                 feature  boruta_rank  importance
0          B_NT - proBNP          1.0    0.085473
1                   LVEF          1.0    0.030783
2               Ea'_mean          1.0    0.030109
3           B_Kreatynina          1.0    0.028327
4            E_diastarea          1.0    0.028255
5                egfr_CG          1.0    0.027774
6               MDRDcorr          1.0    0.026274
7      E_LV2j   rozk cm2          1.0    0.025355
8               Edoeprim          1.0    0.022064
9       E_roz

## Check selection consistency

The table counts how many of the five training folds selected an attribute.
It describes sensitivity to the training sample, not causal confidence.


In [9]:
folds = pd.read_csv(run / "fold_selected_features.csv")
frequency = (
    folds.groupby(["method", "feature"]).size().rename("folds_selected").reset_index()
)
display(
    frequency.sort_values(
        ["method", "folds_selected", "feature"], ascending=[True, False, True]
    )
)


,method,feature,folds_selected
7,boruta,B_Kreatynina,5
9,boruta,B_NT - proBNP,5
12,boruta,E_LV2j rozk cm2,5
14,boruta,E_Lewa komora skurcz,5
16,boruta,E_diastarea,5
...,...,...,...
153,logistic_regression,antiarrh,1
154,logistic_regression,e_lara_sys,1
155,logistic_regression,nitrat,1
157,logistic_regression,sartan,1


## Settings and limitations

Known dates, observation durations, identifiers and `tryb` are excluded. The
remaining columns still need a domain review: their measurement timing and code
meanings are incompletely documented. Follow-up varies between patients, so the
outcome is recorded death, not death within a fixed number of years. Imputation
for these models is training-fitted median/mode, rather than full-data MissForest.


In [10]:
display(config)


{'excluded_columns': ['Data przyjęcia',
  'cid',
  'cid.1',
  'data implantacji',
  'data zgonu lub konca obserwacji',
  'day_count_begin_to_end',
  'tryb',
  'zgon'],
 'missing_target_rows_removed': 0,
 'target': 'zgon',
 'target_coding': {'0': 'no recorded death', '1': 'recorded death'},
 'scope': 'Exploratory association with recorded death, not causal attribution or fixed-horizon risk.',
 'n_features': 20,
 'random_state': 42,
 'primary_metric': 'roc_auc',
 'comparison_rule': 'Higher mean cross-validation ROC AUC; test scores are reporting only.',
 'test_fraction': 0.2,
 'cv_folds': 5,
 'train_rows': 314,
 'test_rows': 79,
 'class_counts': {'1': 226, '0': 167},
 'source': '/home/anka335/Desktop/studia/sem5/Problem Classes I/u-smile_problem_classes/data/raw/cardiac_patient_data.csv',
 'source_sha256': '69a99872287b2b05e08d860c28b44e090d3d2deb6e3ba4b76befadf4cdcc4a30',
 'versions': {'pandas': '3.0.6',
  'numpy': '2.5.4',
  'sklearn': '1.9.1',
  'boruta': '0.4.3'},
 'models': {'logist